# Pronunciation Scoring — Inference Notebook

Run end-to-end pronunciation scoring on any audio file (WAV / MP3 / FLAC / …)  
or a raw NumPy waveform array.

**Pipeline:**
```
Raw audio  →  HuBERT (layer-weighted)  →  Cross-attention fusion (+ Qwen3 text)  →  Transformer block  →  MLP  →  5-dim scores
```

## 1 · Imports

In [2]:
# All heavy imports are encapsulated in notebook_infer/
from notebook_infer import ScoreConfig, load_predictor, score_file, score_array
from notebook_infer.display import show_scores, show_words, show_waveform, play_audio

ModuleNotFoundError: No module named 'inference'

## 2 · Configuration

In [ ]:
cfg = ScoreConfig(
    checkpoint   = "ckpt_hubert_multitask/best.pt",  # set to None for untrained random weights
    device       = "cpu",                             # 'cuda' if GPU is available
    whisper_size = "small",                           # ASR model size: tiny / small / medium
    language     = "en",                              # ISO-639-1 language code
    text_model   = "Qwen/Qwen3-Embedding-0.6B",       # text embedding model
)
print(cfg)

## 3 · Load model

This downloads HuBERT and Qwen3-Embedding on first run (~700 MB total).  
Subsequent runs load from the HuggingFace cache.

In [ ]:
predictor = load_predictor(cfg)
print("Model ready.")

---
## 4A · Score from a file path

In [ ]:
WAV_PATH = "data/learner/01_learner.wav"   # ← change to your audio file

result = score_file(WAV_PATH, predictor, cfg)

---
## 4B · Score from a raw NumPy array

Use this if you already have the waveform in memory (e.g. from `soundfile.read` or `librosa.load`).

In [ ]:
import numpy as np
import soundfile as sf

# Load raw audio into a NumPy array (replace with your own source)
audio_np, sample_rate = sf.read(WAV_PATH, dtype="float32", always_2d=False)
print(f"Audio shape: {audio_np.shape}  |  Sample rate: {sample_rate} Hz")

# Score directly from the array
result = score_array(audio_np, sample_rate, predictor, cfg)

---
## 5 · Display results

In [ ]:
show_scores(result)

In [ ]:
show_words(result)

In [ ]:
show_waveform(WAV_PATH, result)

In [ ]:
play_audio(WAV_PATH)

---
## 6 · Raw JSON output

In [ ]:
import json
print(json.dumps(result, indent=2, ensure_ascii=False))

---
## 7 · Batch scoring

Score multiple files and collect results into a DataFrame.

In [ ]:
import glob
import pandas as pd

audio_files = glob.glob("data/**/*.wav", recursive=True)
print(f"Found {len(audio_files)} audio files.")

rows = []
for path in audio_files:
    r = score_file(path, predictor, cfg)
    rows.append({
        "file":         path,
        "transcript":   r.get("text"),
        "total":        r.get("total"),
        "accuracy":     r.get("accuracy"),
        "fluency":      r.get("fluency"),
        "prosodic":     r.get("prosodic"),
        "completeness": r.get("completeness"),
    })

df = pd.DataFrame(rows)
df